# Comparative SMOTE — reviewer-revised notebook

## Experimental protocol
- Models and hyperparameter spaces are aligned with `ablation-over-ga-stroke-reviewed.ipynb`: Logistic Regression, Decision Tree, Random Forest, and XGBoost.
- Five resampling methods are evaluated: SMOTE, Borderline-SMOTE1, ADASYN, SMOTE-ENN, and SMOTE-TomekLinks.
- Outer validation uses `RepeatedStratifiedKFold(n_splits=10, n_repeats=2, random_state=42)`; GA uses inner CV=3, population size=10, generations=10, MCC fitness, and `n_jobs=-1`.
- Raw dataset is loaded and preprocessing is fitted within the pipeline on each training split. Resampling occurs inside the imbalanced-learn pipeline, so validation/test observations are not resampled.
- Results are reported per model and sampler. No sampler is presumed superior before the new run completes.

## Reviewer-focused outputs
- Per-fold metrics, confusion-matrix counts (including false negatives), model runtime, best GA parameters.
- Mean, SD, and descriptive t-based 95% confidence intervals. Because repeated-CV scores are dependent, these intervals must not be interpreted as corrected independent-sample confidence intervals.
- Model-specific Friedman tests and paired Wilcoxon tests with Holm correction for the five resamplers, for each metric.
- Precision–recall curve points with test-fold prevalence baselines.
- Per-fold resampling class counts and a PCA feature-space diagnostic for the first outer split of each resampler. PCA is exploratory only and does not establish causal mechanisms such as noise removal or reduced class overlap.

## Notes
- The raw Kaggle dataset path below matches the ablation notebook. Update it if your Kaggle dataset mount differs.
- `smote-variants` operates on the preprocessed feature matrix. As in the ablation pipeline, interpolation after one-hot encoding can create fractional indicator values; this limitation should be acknowledged. Do not interpret PCA plots as definitive evidence of class separability.
- This notebook is configured for all models and all resamplers as requested. It is computationally expensive. You may manually split models/resamplers after downloading, but keep the protocol and seeds identical across splits.


In [ ]:
# =====================================================
# INSTALL LIBRARIES (KAGGLE)
# =====================================================
!pip install -q scikit-learn xgboost imbalanced-learn smote-variants sklearn-genetic-opt statsmodels matplotlib

# =====================================================
# IMPORTS
# =====================================================
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import smote_variants as sv

from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    auc, confusion_matrix, balanced_accuracy_score, matthews_corrcoef
)
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Integer, Continuous, Categorical
from scipy.stats import t, friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('default')
SEED = 42
OUTPUT_DIR = Path('.')

# =====================================================
# LOAD RAW DATASET — SAME SOURCE AS ABLATION
# =====================================================
DATA_PATH = '/kaggle/input/datasets/aderahmad/stroke-dataset-raw/healthcare-dataset-stroke-data.csv'
df = pd.read_csv(DATA_PATH)
if 'id' in df.columns:
    df = df.drop(columns=['id'])
if 'stroke' not in df.columns:
    raise ValueError("Kolom target 'stroke' tidak ditemukan. Periksa DATA_PATH dan struktur CSV.")

X = df.drop(columns=['stroke']).copy()
y = df['stroke'].astype(int).copy()

print('Dataset shape:', df.shape)
print('Class counts:', y.value_counts().sort_index().to_dict())
print('Stroke prevalence (PR-AUC no-skill baseline):', round(float(y.mean()), 6))

# =====================================================
# FEATURE GROUPS — MATCH ABLATION NOTEBOOK
# =====================================================
numeric_features = ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
categorical_onehot = ['gender', 'work_type', 'smoking_status']
categorical_ordinal = ['ever_married', 'Residence_type']
required = numeric_features + categorical_onehot + categorical_ordinal
missing_columns = sorted(set(required) - set(X.columns))
if missing_columns:
    raise ValueError(f'Kolom fitur yang dibutuhkan tidak ditemukan: {missing_columns}')

# Tree models: mean imputation + OneHot + ordinal encoding; no scaling.
numeric_transformer_tree = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='mean'))
])
categorical_onehot_transformer_tree = SklearnPipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
categorical_ordinal_transformer_tree = SklearnPipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['No', 'Yes'], ['Rural', 'Urban']],
        handle_unknown='use_encoded_value', unknown_value=-1
    ))
])
preprocessor_tree = ColumnTransformer([
    ('num', numeric_transformer_tree, numeric_features),
    ('cat_onehot', categorical_onehot_transformer_tree, categorical_onehot),
    ('cat_ordinal', categorical_ordinal_transformer_tree, categorical_ordinal)
], remainder='drop')

# Linear/SVM models: same encodings and mean imputation, plus StandardScaler on numeric features.
numeric_transformer_linear = SklearnPipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])
categorical_onehot_transformer_linear = SklearnPipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
categorical_ordinal_transformer_linear = SklearnPipeline([
    ('ordinal', OrdinalEncoder(
        categories=[['No', 'Yes'], ['Rural', 'Urban']],
        handle_unknown='use_encoded_value', unknown_value=-1
    ))
])
preprocessor_linear = ColumnTransformer([
    ('num', numeric_transformer_linear, numeric_features),
    ('cat_onehot', categorical_onehot_transformer_linear, categorical_onehot),
    ('cat_ordinal', categorical_ordinal_transformer_linear, categorical_ordinal)
], remainder='drop')

# =====================================================
# BASE MODELS — SAME AS ABLATION NOTEBOOKS
# =====================================================
base_models = {
    'Random Forest': RandomForestClassifier(random_state=SEED),
    'XGBoost': XGBClassifier(random_state=SEED, eval_metric='logloss'),
    'Logistic Regression': LogisticRegression(random_state=SEED, max_iter=1000),
    'SVM': SVC(probability=True, random_state=SEED)
}

# =====================================================
# HYPERPARAMETER SEARCH SPACES — COPIED FROM ABLATION
# =====================================================
rf_param_grid = {
    'model__n_estimators': Integer(50, 300),
    'model__max_depth': Integer(3, 10),
    'model__min_samples_split': Integer(2, 10),
    'model__min_samples_leaf': Integer(5, 10),
    'model__class_weight': Categorical(['balanced', 'balanced_subsample', None])
}
xgb_param_grid = {
    'model__n_estimators': Integer(50, 300),
    'model__max_depth': Integer(3, 10),
    'model__learning_rate': Continuous(0.01, 0.3),
    'model__subsample': Continuous(0.5, 1.0),
    'model__colsample_bytree': Continuous(0.5, 1.0),
    'model__scale_pos_weight': Continuous(1, 25)
}
lr_param_grid = {
    'model__C': Continuous(0.01, 10.0, distribution='log-uniform'),
    'model__penalty': Categorical(['l2']),
    'model__solver': Categorical(['lbfgs', 'saga']),
    'model__class_weight': Categorical([None, 'balanced'])
}
svm_param_grid = {
    'model__C': Continuous(0.01, 10.0, distribution='log-uniform'),
    'model__kernel': Categorical(['rbf']),
    'model__gamma': Categorical(['scale', 'auto']),
    'model__class_weight': Categorical([None, 'balanced'])
}
param_grids = {
    'Random Forest': rf_param_grid,
    'XGBoost': xgb_param_grid,
    'Logistic Regression': lr_param_grid,
    'SVM': svm_param_grid
}
preprocessors = {
    'Random Forest': preprocessor_tree,
    'XGBoost': preprocessor_tree,
    'Logistic Regression': preprocessor_linear,
    'SVM': preprocessor_linear
}

# =====================================================
# SMOTE-VARIANTS ADAPTER
# Supports smote-variants sampler API inside imbalanced-learn Pipeline.
# Records resampling counts on the final outer-training refit.
# =====================================================
class SmoteVariantsAdapter(BaseEstimator):
    def __init__(self, variant_name='SMOTE', random_state=42):
        self.variant_name = variant_name
        self.random_state = random_state

    def fit_resample(self, X, y):
        sampler_map = {
            'SMOTE': sv.SMOTE,
            'Borderline-SMOTE': sv.Borderline_SMOTE1,
            'ADASYN': sv.ADASYN,
            'SMOTE-ENN': sv.SMOTE_ENN,
            'SMOTE-Tomek': sv.SMOTE_TomekLinks
        }
        if self.variant_name not in sampler_map:
            raise ValueError(f'Varian SMOTE tidak dikenal: {self.variant_name}')
        sampler = sampler_map[self.variant_name](random_state=self.random_state)
        y_arr = np.asarray(y).astype(int)
        self.n_samples_before_ = int(len(y_arr))
        self.class_counts_before_ = pd.Series(y_arr).value_counts().sort_index().to_dict()
        X_resampled, y_resampled = sampler.sample(np.asarray(X), y_arr)
        y_resampled = np.asarray(y_resampled).astype(int)
        self.n_samples_after_ = int(len(y_resampled))
        self.class_counts_after_ = pd.Series(y_resampled).value_counts().sort_index().to_dict()
        return np.asarray(X_resampled), y_resampled

# All variants remain in this notebook; user will split manually if needed.
SMOTE_VARIANTS = ['SMOTE','Borderline-SMOTE', 'ADASYN', 'SMOTE-ENN', 'SMOTE-Tomek']
MODEL_NAMES = ['Random Forest', 'XGBoost', 'Logistic Regression', 'SVM']

# =====================================================
# PIPELINE FACTORY — PREPROCESS -> RESAMPLE -> MODEL
# =====================================================
def make_pipeline(model_name, variant_name):
    return Pipeline([
        ('preprocessor', preprocessors[model_name]),
        ('smote', SmoteVariantsAdapter(variant_name=variant_name, random_state=SEED)),
        ('model', base_models[model_name])
    ])

# =====================================================
# EXPERIMENT SETTINGS — SAME AS ABLATION
# =====================================================
OUTER_CV = RepeatedStratifiedKFold(n_splits=10, n_repeats=2, random_state=SEED)
OUTER_SPLITS = list(OUTER_CV.split(X, y))
INNER_CV = 3
GA_POPULATION_SIZE = 10
GA_GENERATIONS = 10
GA_SCORING = 'matthews_corrcoef'
GA_N_JOBS = -1

print(f'Outer folds: {len(OUTER_SPLITS)}')
print(f'Planned GA searches: {len(OUTER_SPLITS) * len(MODEL_NAMES) * len(SMOTE_VARIANTS)}')
print('CAUTION: This is a computationally expensive experiment; five models × five variants × 20 outer folds.')


In [ ]:
# =====================================================
# OUTPUT FILES / CHECKPOINTS
# =====================================================
# This cell starts a fresh run and clears prior outputs with these names.
# Do not rerun this cell mid-experiment, or prior checkpoint CSVs will be deleted.
OUTPUT_FILES = {
    'fold': 'comparative_smote_fold_results.csv',
    'pred': 'comparative_smote_predictions.csv',
    'runtime': 'comparative_smote_runtime_results.csv',
    'params': 'comparative_smote_best_parameters.csv',
    'resampling': 'comparative_smote_resampling_counts.csv',
    'summary': 'comparative_smote_summary_results.csv',
    'stats': 'comparative_smote_statistical_tests.csv',
}
for path in OUTPUT_FILES.values():
    if os.path.exists(path):
        os.remove(path)

METRICS = [
    'accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc',
    'average_precision', 'pr_auc', 'balanced_accuracy', 'mcc'
]

fold_rows = []
prediction_rows = []
runtime_rows = []
parameter_rows = []
resampling_rows = []

# =====================================================
# OUTER CV: EVERY MODEL × EVERY SMOTE VARIANT
# =====================================================
for variant_name in SMOTE_VARIANTS:
    print('\n' + '#' * 90)
    print(f'COMPARATIVE VARIANT: {variant_name}')
    print('#' * 90)

    for model_name in MODEL_NAMES:
        print(f'\n### Model: {model_name} | Variant: {variant_name}')
        for fold, (train_idx, test_idx) in enumerate(OUTER_SPLITS, start=1):
            repeat = ((fold - 1) // 10) + 1
            fold_in_repeat = ((fold - 1) % 10) + 1
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
            before = y_train.value_counts().sort_index().to_dict()

            estimator = make_pipeline(model_name, variant_name)
            ga_search = GASearchCV(
                estimator=estimator,
                cv=INNER_CV,
                scoring=GA_SCORING,
                population_size=GA_POPULATION_SIZE,
                generations=GA_GENERATIONS,
                param_grid=param_grids[model_name],
                n_jobs=GA_N_JOBS,
                verbose=False
            )

            start_time = time.perf_counter()
            ga_search.fit(X_train, y_train)
            runtime_seconds = time.perf_counter() - start_time
            best_model = ga_search.best_estimator_
            y_pred = best_model.predict(X_test)
            y_prob = best_model.predict_proba(X_test)[:, 1]

            tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
            specificity = tn / (tn + fp) if (tn + fp) else 0.0
            pr_precision, pr_recall, _ = precision_recall_curve(y_test, y_prob)
            pr_auc_value = auc(pr_recall, pr_precision)
            metrics = {
                'accuracy': accuracy_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred, zero_division=0),
                'recall': recall_score(y_test, y_pred, zero_division=0),
                'specificity': specificity,
                'f1': f1_score(y_test, y_pred, zero_division=0),
                'roc_auc': roc_auc_score(y_test, y_prob),
                'average_precision': average_precision_score(y_test, y_prob),
                'pr_auc': pr_auc_value,
                'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
                'mcc': matthews_corrcoef(y_test, y_pred),
            }
            sampler_fit = best_model.named_steps['smote']
            after = getattr(sampler_fit, 'class_counts_after_', {})
            after_majority = int(after.get(0, 0))
            after_minority = int(after.get(1, 0))

            row = {
                'Variant': variant_name,
                'Model': model_name,
                'Outer_Fold': fold,
                'Repeat': repeat,
                'Fold_in_Repeat': fold_in_repeat,
                'Train_Majority_Before': int(before.get(0, 0)),
                'Train_Minority_Before': int(before.get(1, 0)),
                'Train_Majority_After': after_majority,
                'Train_Minority_After': after_minority,
                'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
                'Runtime_Seconds': runtime_seconds,
                'Best_Params_JSON': json.dumps(ga_search.best_params_, sort_keys=True),
                **metrics
            }
            fold_rows.append(row)
            runtime_rows.append({
                'Variant': variant_name, 'Model': model_name, 'Outer_Fold': fold,
                'Repeat': repeat, 'Runtime_Seconds': runtime_seconds
            })
            parameter_rows.append({
                'Variant': variant_name, 'Model': model_name, 'Outer_Fold': fold,
                'Repeat': repeat, **ga_search.best_params_
            })
            resampling_rows.append({
                'Variant': variant_name, 'Model': model_name, 'Outer_Fold': fold,
                'Repeat': repeat,
                'Before_Class_0': int(before.get(0, 0)),
                'Before_Class_1': int(before.get(1, 0)),
                'After_Class_0': after_majority,
                'After_Class_1': after_minority,
                'Before_Total': int(len(y_train)),
                'After_Total': int(sampler_fit.n_samples_after_) if hasattr(sampler_fit, 'n_samples_after_') else None
            })
            for idx, yt, yp, prob in zip(test_idx, y_test.to_numpy(), y_pred, y_prob):
                prediction_rows.append({
                    'Variant': variant_name, 'Model': model_name,
                    'Outer_Fold': fold, 'Repeat': repeat,
                    'Row_Index': int(idx), 'y_true': int(yt),
                    'y_pred': int(yp), 'y_prob_stroke': float(prob)
                })

            # Write checkpoints after every model × variant × fold.
            pd.DataFrame([row]).to_csv(OUTPUT_FILES['fold'], mode='a', index=False,
                                       header=not os.path.exists(OUTPUT_FILES['fold']))
            pd.DataFrame([runtime_rows[-1]]).to_csv(OUTPUT_FILES['runtime'], mode='a', index=False,
                                       header=not os.path.exists(OUTPUT_FILES['runtime']))
            pd.DataFrame([parameter_rows[-1]]).to_csv(OUTPUT_FILES['params'], mode='a', index=False,
                                       header=not os.path.exists(OUTPUT_FILES['params']))
            pd.DataFrame([resampling_rows[-1]]).to_csv(OUTPUT_FILES['resampling'], mode='a', index=False,
                                       header=not os.path.exists(OUTPUT_FILES['resampling']))
            pred_batch = pd.DataFrame(prediction_rows[-len(test_idx):])
            pred_batch.to_csv(OUTPUT_FILES['pred'], mode='a', index=False,
                              header=not os.path.exists(OUTPUT_FILES['pred']))

            print(
                f'Repeat {repeat}, fold {fold_in_repeat}/10 | '
                f'BalAcc={metrics["balanced_accuracy"]:.4f} | '
                f'MCC={metrics["mcc"]:.4f} | PR-AUC={metrics["pr_auc"]:.4f} | '
                f'FN={fn} | runtime={runtime_seconds:.1f}s'
            )

print('\nAll model × variant × outer-fold combinations completed.')


In [ ]:
# =====================================================
# SUMMARY: MEAN, SD, AND DESCRIPTIVE 95% CI
# =====================================================
fold_df = pd.read_csv(OUTPUT_FILES['fold'])
summary_rows = []
for (variant_name, model_name), group in fold_df.groupby(['Variant', 'Model'], sort=False):
    for metric in METRICS:
        values = group[metric].dropna().to_numpy(dtype=float)
        n = len(values)
        mean_value = float(np.mean(values))
        sd_value = float(np.std(values, ddof=1)) if n > 1 else float('nan')
        if n > 1:
            margin = float(t.ppf(0.975, df=n-1) * sd_value / np.sqrt(n))
            ci_low, ci_high = mean_value - margin, mean_value + margin
        else:
            ci_low, ci_high = float('nan'), float('nan')
        summary_rows.append({
            'Variant': variant_name, 'Model': model_name, 'Metric': metric,
            'N_Outer_Fold_Scores': n, 'Mean': mean_value, 'SD': sd_value,
            'CI95_Lower_Descriptive': ci_low, 'CI95_Upper_Descriptive': ci_high,
            'CI_Note': 'Descriptive t interval; repeated-CV fold scores are dependent.'
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_FILES['summary'], index=False)

# =====================================================
# MODEL-SPECIFIC STATISTICS: FRIEDMAN + PAIRED WILCOXON/HOLM
# Scores are paired by identical outer split; repeated-CV dependence remains a limitation.
# =====================================================
stat_rows = []
comparison_metrics = ['balanced_accuracy', 'mcc', 'pr_auc']
variants_in_order = SMOTE_VARIANTS
for model_name in MODEL_NAMES:
    model_data = fold_df[fold_df['Model'] == model_name]
    for metric in comparison_metrics:
        pivot = model_data.pivot(index='Outer_Fold', columns='Variant', values=metric)
        if not all(v in pivot.columns for v in variants_in_order):
            continue
        pivot = pivot[variants_in_order].dropna()
        if len(pivot) < 2:
            continue
        try:
            stat, p_value = friedmanchisquare(*[pivot[v].to_numpy() for v in variants_in_order])
            stat_rows.append({
                'Model': model_name, 'Metric': metric, 'Test': 'Friedman',
                'Comparison': 'All five variants', 'Statistic': float(stat),
                'P_Value': float(p_value), 'P_Value_Holm': np.nan,
                'N_Paired_Outer_Folds': len(pivot),
                'Interpretation_Note': 'Exploratory fold-level test; repeated-CV scores are dependent.'
            })
        except Exception as exc:
            stat_rows.append({
                'Model': model_name, 'Metric': metric, 'Test': 'Friedman',
                'Comparison': 'All five variants', 'Statistic': np.nan,
                'P_Value': np.nan, 'P_Value_Holm': np.nan,
                'N_Paired_Outer_Folds': len(pivot), 'Interpretation_Note': f'Test failed: {exc}'
            })
        pair_records = []
        for i in range(len(variants_in_order)):
            for j in range(i + 1, len(variants_in_order)):
                a, b = variants_in_order[i], variants_in_order[j]
                try:
                    w_stat, p_pair = wilcoxon(pivot[a], pivot[b], zero_method='wilcox', alternative='two-sided')
                except ValueError:
                    w_stat, p_pair = np.nan, 1.0
                pair_records.append((a, b, w_stat, p_pair))
        valid_p = [r[3] for r in pair_records]
        adjusted = multipletests(valid_p, method='holm')[1] if valid_p else []
        for rec, p_adj in zip(pair_records, adjusted):
            a, b, w_stat, p_pair = rec
            stat_rows.append({
                'Model': model_name, 'Metric': metric, 'Test': 'Wilcoxon signed-rank + Holm',
                'Comparison': f'{a} vs {b}', 'Statistic': float(w_stat) if pd.notna(w_stat) else np.nan,
                'P_Value': float(p_pair), 'P_Value_Holm': float(p_adj),
                'N_Paired_Outer_Folds': len(pivot),
                'Interpretation_Note': 'Exploratory fold-level paired test; repeated-CV scores are dependent.'
            })
stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(OUTPUT_FILES['stats'], index=False)

print('Summary saved:', OUTPUT_FILES['summary'])
print('Statistical tests saved:', OUTPUT_FILES['stats'])


In [ ]:
# =====================================================
# PRECISION-RECALL CURVES WITH PREVALENCE BASELINE
# Curves pool out-of-fold predictions from repeated CV; each row may appear once per repeat.
# Use as descriptive visualization, not as independent-sample inference.
# =====================================================
pred_df = pd.read_csv(OUTPUT_FILES['pred'])
prevalence = float(y.mean())
for model_name in MODEL_NAMES:
    fig, ax = plt.subplots(figsize=(8, 6))
    for variant_name in SMOTE_VARIANTS:
        subset = pred_df[(pred_df['Model'] == model_name) & (pred_df['Variant'] == variant_name)]
        if subset.empty:
            continue
        precision_curve, recall_curve, _ = precision_recall_curve(
            subset['y_true'].to_numpy(), subset['y_prob_stroke'].to_numpy()
        )
        pooled_ap = average_precision_score(subset['y_true'], subset['y_prob_stroke'])
        ax.plot(recall_curve, precision_curve, label=f'{variant_name} (AP={pooled_ap:.3f})')
    ax.axhline(prevalence, linestyle='--', label=f'Prevalence baseline={prevalence:.3f}')
    ax.set_title(f'Precision–Recall Curves — {model_name}')
    ax.set_xlabel('Recall (Sensitivity)')
    ax.set_ylabel('Precision')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    safe_model = model_name.lower().replace(' ', '_')
    fig.savefig(f'comparative_smote_pr_curve_{safe_model}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# =====================================================
# FINAL SANITY CHECKS
# =====================================================
expected = len(SMOTE_VARIANTS) * len(MODEL_NAMES) * len(OUTER_SPLITS)
actual = len(pd.read_csv(OUTPUT_FILES['fold']))
print(f'Expected model × variant × fold records: {expected}')
print(f'Observed records in fold CSV: {actual}')
print('Output files:')
for key, path in OUTPUT_FILES.items():
    print(f'- {key}: {path}')
print('- PR curves: comparative_smote_pr_curve_<model>.png')
if actual != expected:
    print('WARNING: result count differs from expected; inspect any interruption or failed combination.')
